<a href="https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/Bot%2BServer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q aiogram fastapi uvicorn nest_asyncio aiohttp python-multipart torch transformers pymorphy2

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 859.1/859.1 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 51.3 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

BOT_TOKEN = userdata.get('BOT_TOKEN')
if not BOT_TOKEN or BOT_TOKEN == "YOUR_BOT_TOKEN_HERE":
    raise ValueError("ОШИБКА: Токен не найден! Добавь секрет 'BOT_TOKEN' в настройки Colab (значок ключа слева).")
else:
    print(" Токен успешно загружен!")

import torch
from transformers import pipeline
import os
import uuid
import aiohttp
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
import nest_asyncio
import uvicorn
import threading
import time
from aiogram import Bot, Dispatcher, F
from aiogram.types import Message
from aiogram.filters import Command

 Токен успешно загружен!


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔧 Используем устройство: {device}")
print("⏳ Загрузка модели Whisper...")
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small",
    device=device,
    # ⚠️ ВАЖНО: Явно указываем русский язык, чтобы не было случайных языков
    generate_kwargs={"language": "russian", "task": "transcribe"}
)

def transcribe_audio(file_path: str) -> str:
    try:
        result = asr_pipeline(file_path)
        return result["text"].strip()
    except Exception as e:
        print(f" Ошибка ASR: {e}")
        return ""

# МОДЕЛЬ КЛАССИФИКАЦИИ (3 КЛАССА)
print("⏳ Загрузка модели классификации...")

# ⚠️ ВАЖНО: Укажи здесь путь к папке, куда ты сохранила свою лучшую модель в Ячейке 20!
# Например: "./final_models/best_rubert-base" или "./augmented_models/best_augmented_rubert-base"
CUSTOM_MODEL_PATH = "./final_models/best_model"

# Если твоя папка не найдена, используем публичную 3-классовую модель для демо
if not os.path.exists(CUSTOM_MODEL_PATH):
    print("⚠️ Папка с твоей моделью не найдена. Использую публичную 3-классовую модель (rusentiment).")
    MODEL_TO_USE = "blanchefort/rubert-base-cased-sentiment-rusentiment"
else:
    print(f"✅ Найдена твоя обученная модель: {CUSTOM_MODEL_PATH}")
    MODEL_TO_USE = CUSTOM_MODEL_PATH

# Используем text-classification, он лучше работает с 3+ классами, чем sentiment-analysis
sentiment_pipeline = pipeline(
    "text-classification",
    model=MODEL_TO_USE,
    device=device,
    top_k=None # Возвращает вероятности всех классов, а не только лучший
)

# ⚠️ ВАЖНО: Проверь, как твоя модель называет классы!
# Если это твоя модель, там скорее всего 'LABEL_0', 'LABEL_1', 'LABEL_2'
# Если публичная rusentiment, там 'NEGATIVE', 'NEUTRAL', 'POSITIVE'
# Отредактируй этот словарь под свои реалии:
LABEL_MAPPING = {
    "LABEL_0": "negative",   # Или "NEGATIVE"
    "LABEL_1": "neutral",    # Или "NEUTRAL"
    "LABEL_2": "positive",   # Или "POSITIVE"
    "NEGATIVE": "negative",
    "NEUTRAL": "neutral",
    "POSITIVE": "positive"
}

SENTIMENT_NAMES_RU = {
    "negative": "Негативный",
    "neutral": "Нейтральный",
    "positive": "Позитивный",
    "unknown": "Не определен"
}

SENTIMENT_EMOJIS = {
    "negative": "😡",
    "neutral": "😐",
    "positive": "😊",
    "unknown": "❓"
}

def classify_sentiment(text: str) -> dict:
    """Классификация тональности через 3-классовую модель."""
    if not text or not text.strip():
        return {"sentiment": "unknown", "confidence": 0.0}

    try:
        # Обрезаем текст до 512 символов (стандартный лимит BERT)
        result = sentiment_pipeline(text[:512])[0]

        # result выглядит так: [{'label': 'LABEL_1', 'score': 0.85}, ...]
        # Берем класс с максимальной вероятностью (он всегда первый в списке)
        best_prediction = result[0]
        label = best_prediction["label"]
        confidence = best_prediction["score"]

        # Преобразуем метку модели в наш формат
        sentiment = LABEL_MAPPING.get(label, "unknown")

        return {"sentiment": sentiment, "confidence": confidence}
    except Exception as e:
        print(f"❌ Ошибка классификации: {e}")
        return {"sentiment": "unknown", "confidence": 0.0}

print("Все модели успешно загружены!")


🔧 Используем устройство: cpu
⏳ Загрузка модели Whisper...


config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

... (13 outputs left out)

tokenizer_config.json:   0%|          | 0.00/495 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.40M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Все модели успешно загружены!


In [ ]:
from fastapi import FastAPI, File, UploadFile
from fastapi.responses import JSONResponse
import uuid

app = FastAPI(title="NLP Sentiment Server")

@app.post("/predict")
async def predict_endpoint(file: UploadFile = File(...)):
    # 1. Генерируем уникальное имя для временного файла
    file_extension = file.filename.split(".")[-1]
    temp_filename = f"temp_audio_{uuid.uuid4().hex}.{file_extension}"

    # 2. Сохраняем полученное аудио
    with open(temp_filename, "wb") as buffer:
        buffer.write(await file.read())

    try:
        # 3. Распознаем речь (ASR)
        text = transcribe_audio(temp_filename)

        # 4. Классифицируем текст
        result = classify_sentiment(text)

        # 5. Возвращаем красивый JSON
        return JSONResponse(content={
            "text": text,
            "sentiment": result["sentiment"],
            "confidence": result["confidence"]
        })

    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})

    finally:
        # 6. Обязательно удаляем временный файл, чтобы не забить память Colab
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

In [ ]:
app = FastAPI(title="NLP Sentiment Server")

@app.post("/predict")
async def predict_endpoint(file: UploadFile = File(...)):
    file_extension = file.filename.split(".")[-1]
    temp_filename = f"temp_audio_{uuid.uuid4().hex}.{file_extension}"

    with open(temp_filename, "wb") as buffer:
        buffer.write(await file.read())

    try:
        text = transcribe_audio(temp_filename)
        result = classify_sentiment(text)

        return JSONResponse(content={
            "text": text,
            "sentiment": result["sentiment"],
            "confidence": result["confidence"]
        })
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})
    finally:
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

# Запуск сервера в фоне
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("FastAPI сервер запущен в фоне на http://127.0.0.1:8000")

FastAPI сервер запущен в фоне на http://127.0.0.1:8000


In [ ]:
@dp.message(F.voice)
async def handle_voice(message: Message):
    status_msg = await message.answer("🎙️ Слушаю и анализирую...")
    file_id = message.voice.file_id
    local_filename = f"bot_temp_{message.from_user.id}.ogg"

    try:
        file = await bot.get_file(file_id2)
        await bot.download_file(file.file_path, local_filename)

        async with aiohttp.ClientSession() as session:
            with open(local_filename, "rb") as audio_file:
                form_data = aiohttp.FormData()
                form_data.add_field("file", audio_file, filename=local_filename, content_type="audio/ogg")

                async with session.post("http://127.0.0.1:8000/predict", data=form_data) as response:
                    if response.status == 200:
                        result = await response.json()
                        text = result.get("text", "Не удалось распознать речь")
                        sentiment = result.get("sentiment", "unknown")
                        confidence = result.get("confidence", 0.0)

                        emoji = SENTIMENT_EMOJIS.get(sentiment, "❓")
                        sentiment_ru = SENTIMENT_NAMES_RU.get(sentiment, "Не определен")

                        reply = (
                            f"{emoji} *Результат анализа:*\n\n"
                            f"📝 *Текст:* {text}\n"
                            f"📊 *Тональность:* {sentiment_ru}\n"
                            f"🎯 *Уверенность:* {confidence:.1%}"
                        )
                        await status_msg.edit_text(reply, parse_mode="Markdown")
                    else:
                        error_text = await response.text()
                        await status_msg.edit_text(f"❌ Ошибка сервера: {response.status}\n{error_text}")

    except Exception as e:
        await status_msg.edit_text(f"❌ Произошла ошибка: {str(e)}")
            finally:
        if os.path.exists(local_filename):
            os.remove(local_filename)

print("🤖 Бот запущен и ожидает сообщения...")
print("Чтобы остановить бота — нажми квадратик (Stop) слева от ячейки.")

# Запуск поллинга
await dp.start_polling(bot)

IndentationError: unexpected indent (2866140727.py, line 39)

Детекция Out of Distribution (OOD)

Поскольку система работает с результатами ASR (Whisper), на вход классификатору может попасть "мусор": фоновый шум, галлюцинации модели распознавания речи или текст на другом языке.
Чтобы модель не выдавала ложноположительные результаты с высокой уверенностью, мы внедрили механизм OOD-детекции на этапе инференса, основанный на двух метриках:
Порог уверенности (Confidence Threshold): Если max(softmax(logits)) < 0.55, запрос отвергается.
Энтропия предсказания (Predictive Entropy): Если энтропия распределения вероятностей превышает 0.95 (при максимуме ~1.098 для 3 классов), это сигнализирует о том, что модель "колеблется" между классами, что характерно для данных вне обучающего распределения.
Результат: Бот теперь корректно отвечает "Не определено" на нерелевантные входные данные, что повышает доверие пользователей и снижает количество критических ошибок.